# Validate Data With gx_framework

This notebook is the primary notebook entry point for the simplified Great Expectations framework.

It demonstrates:
- loading sample data into Spark
- validating a Spark DataFrame with `validate_dataframe(...)`
- validating a Spark table with `validate_table(...)`
- interpreting the returned result contract
- locating logs and saved validation outputs

Prerequisites:
- a Spark session is available in the current notebook environment
- the repository `src/`, `gx/`, `config/`, and `logs/` folders are available
- the sample dataset and expectation suites in this repository are present

This notebook keeps framework logic in importable Python modules and uses notebook cells only for orchestration and explanation.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

from pyspark.sql import SparkSession

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from gx_framework import validate_dataframe, validate_table

spark = SparkSession.getActiveSession() or SparkSession.builder.appName(
    "gx_framework_demo"
).getOrCreate()

DATASET_NAME = "green_tripdata_2017"
TABLE_NAME = "demo_green_tripdata_2017"
DATA_PATH = REPO_ROOT / "data" / "green_tripdata_2017_sample.csv"
LOGS_ROOT = REPO_ROOT / "logs"

print(f"Repository root: {REPO_ROOT}")
print(f"Sample data path: {DATA_PATH}")
print(f"Logs root: {LOGS_ROOT}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/08 14:22:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Repository root: /workspaces/great-expectations
Sample data path: /workspaces/great-expectations/data/green_tripdata_2017_sample.csv
Logs root: /workspaces/great-expectations/logs


## Load Sample Data

This section uses the repository sample CSV so the notebook exercises real project assets instead of synthetic data.

The dataset name `green_tripdata_2017` matches the mapping in `config/datasets.yml`, which resolves to the existing expectation suite `bronze.sales.green_tripdata_2017`.

In [2]:
df = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .csv(str(DATA_PATH))
)

df.printSchema()
print(f"Row count: {df.count()}")
print(f"First 10 columns: {df.columns[:10]}")

root
 |-- vendorID: integer (nullable = true)
 |-- paymentType: integer (nullable = true)
 |-- passengerCount: integer (nullable = true)
 |-- tripDistance: double (nullable = true)
 |-- totalAmount: double (nullable = true)

Row count: 10
First 10 columns: ['vendorID', 'paymentType', 'passengerCount', 'tripDistance', 'totalAmount']


26/03/08 14:22:52 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


## Validate A DataFrame

Use the minimal public API when you already have a Spark DataFrame in memory.

This example relies on dataset-driven suite resolution, so the caller does not need to reference Great Expectations context objects, datasources, assets, or validators directly.

In [3]:
dataframe_result = validate_dataframe(
    df=df,
    dataset_name=DATASET_NAME,
    save_results=True,
    log_level="INFO",
)

print(json.dumps(dataframe_result, indent=2))

{"timestamp_utc": "2026-03-08T14:22:53.119432Z", "level": "INFO", "logger": "gx_framework.suite_resolver", "event": "suite_resolution", "taskName": "Task-45", "resolution_path": "config_mapping", "dataset_name": "green_tripdata_2017", "suite_name": "bronze.sales.green_tripdata_2017"}
{"timestamp_utc": "2026-03-08T14:22:53.225327Z", "level": "INFO", "logger": "gx_framework", "event": "validation_started", "taskName": "Task-45", "dataset_name": "green_tripdata_2017", "suite_name": "bronze.sales.green_tripdata_2017", "run_name": "green_tripdata_2017_20260308_142253", "row_count": 10}
Calculating Metrics: 100%|██████████| 23/23 [00:00<00:00, 27.76it/s]
{"timestamp_utc": "2026-03-08T14:22:57.291179Z", "level": "INFO", "logger": "gx_framework", "event": "validation_result_saved", "taskName": "Task-45", "dataset_name": "green_tripdata_2017", "suite_name": "bronze.sales.green_tripdata_2017", "run_name": "green_tripdata_2017_20260308_142253", "result_path": "/workspaces/great-expectations/logs/

{
  "success": true,
  "dataset_name": "green_tripdata_2017",
  "suite_name": "bronze.sales.green_tripdata_2017",
  "run_name": "green_tripdata_2017_20260308_142253",
  "total_expectations": 3,
  "successful_expectations": 3,
  "failed_expectations": 0,
  "success_percent": 100.0,
  "validation_time_utc": "2026-03-08T14:22:57Z",
  "failure_details": []
}


## Validate A Table

Use `validate_table(...)` when the dataset is already registered in the active Spark session.

The function loads the table with `spark.table(table_name)` and delegates to the same validation flow used for DataFrames.

In [4]:
df.createOrReplaceTempView(TABLE_NAME)

table_result = validate_table(
    table_name=TABLE_NAME,
    dataset_name=DATASET_NAME,
    save_results=False,
    log_level="INFO",
)

print(json.dumps(table_result, indent=2))

{"timestamp_utc": "2026-03-08T14:22:57.479001Z", "level": "INFO", "logger": "gx_framework.suite_resolver", "event": "suite_resolution", "taskName": "Task-48", "resolution_path": "config_mapping", "dataset_name": "green_tripdata_2017", "suite_name": "bronze.sales.green_tripdata_2017"}
{"timestamp_utc": "2026-03-08T14:22:57.605761Z", "level": "INFO", "logger": "gx_framework", "event": "validation_started", "taskName": "Task-48", "dataset_name": "green_tripdata_2017", "suite_name": "bronze.sales.green_tripdata_2017", "run_name": "green_tripdata_2017_20260308_142257", "row_count": 10}
26/03/08 14:22:57 WARN CacheManager: Asked to cache already cached data.
Calculating Metrics: 100%|██████████| 23/23 [00:00<00:00, 47.26it/s] 
{"timestamp_utc": "2026-03-08T14:22:58.157938Z", "level": "INFO", "logger": "gx_framework", "event": "validation_finished", "taskName": "Task-48", "dataset_name": "green_tripdata_2017", "suite_name": "bronze.sales.green_tripdata_2017", "run_name": "green_tripdata_2017_

{
  "success": true,
  "dataset_name": "green_tripdata_2017",
  "suite_name": "bronze.sales.green_tripdata_2017",
  "run_name": "green_tripdata_2017_20260308_142257",
  "total_expectations": 3,
  "successful_expectations": 3,
  "failed_expectations": 0,
  "success_percent": 100.0,
  "validation_time_utc": "2026-03-08T14:22:58Z",
  "failure_details": []
}


## Interpreting Results And Maintaining This Notebook

The returned dictionary is designed for notebooks and pipeline orchestration.

Key fields:
- `success` indicates whether the suite passed
- `suite_name` shows the resolved expectation suite
- `failed_expectations` and `failure_details` summarize validation issues
- `validation_time_utc` supports operational logging and audit trails

Operational notes:
- log files are written to `logs/gx_validation_YYYYMMDD.log`
- saved result payloads are written under `logs/validation_results/` when `save_results=True`
- for pipeline fail-fast behavior, call `validate_dataframe(..., fail_on_error=True)` or `validate_table(..., fail_on_error=True)`

Maintenance notes:
- restart the kernel if notebook imports become stale after local module edits
- keep framework logic in `src/` modules rather than expanding notebook utility code
- prefer the CSV-based demo path in this notebook if Delta support is unavailable in the current environment
- the notebooks in `notebooks/archive/` are historical references and are no longer the primary documented path

In [5]:
summary = {
    "dataframe_success": dataframe_result["success"],
    "table_success": table_result["success"],
    "suite_name": dataframe_result["suite_name"],
    "failed_expectations": dataframe_result["failed_expectations"],
    "saved_result_files": sorted(
        path.name for path in (LOGS_ROOT / "validation_results").glob("*.json")
    ) if (LOGS_ROOT / "validation_results").exists() else [],
}

print(json.dumps(summary, indent=2))

if dataframe_result["failure_details"]:
    print("Failure details:")
    print(json.dumps(dataframe_result["failure_details"], indent=2))

{
  "dataframe_success": true,
  "table_success": true,
  "suite_name": "bronze.sales.green_tripdata_2017",
  "failed_expectations": 0,
  "saved_result_files": [
    "green_tripdata_2017_20260308_142253.json"
  ]
}
